In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
df = pd.read_csv("../../data/processed/fig/fig2_modal_compare.csv")
df

In [ ]:
df_melted = df.melt(id_vars=['SDG'], 
                    value_vars=['r2_sv', 'r2_rs', 'r2'], 
                    var_name='Metric', 
                    value_name='R2_Score')
df_melted

In [ ]:
df_melted.groupby(['SDG', 'Metric'])['R2_Score'].describe()

In [ ]:
df_grouped = df_melted.groupby(['SDG', 'Metric']).mean().reset_index()
df_grouped

In [ ]:
# Grouping rule: if Fusion gains > 0.02 over Single-Modality, treat as Synergy.
synergy_group = ['4', '5', '8', '10', '11']  # extended to include 9 and 8
micro_group   = ['3', '6','9', '16']                        # SV-dominant SDGs with small Fusion gain
macro_group   = ['1', '13']                 # RS-dominant SDGs

# Concat + sort.
plot_order = synergy_group + micro_group + macro_group

# Rename legend entries.
label_map = {
    'r2': 'Fusion (Ours)',
    'r2_sv': 'Street view only',
    'r2_rs': 'Satellite only'
}
df_plot = df_melted.copy()
df_plot['Metric'] = df_plot['Metric'].map(label_map)

In [ ]:
# Tall-narrow canvas for the horizontal bar chart.
fig, ax = plt.subplots(figsize=(8, 12))

custom_palette = {
    'Fusion (Ours)':    '#8E8BFE',
    'Street view only': '#5FB7B9',  
    'Satellite only':   '#F9C74F'   
}

# Key change: swap x and y.
ax = sns.barplot(
    data=df_plot,
    y='SDG',          # y is the category
    x='R2_Score',     # x is the value
    hue='Metric',
    order=plot_order,
    palette=custom_palette,
    # edgecolor='black',
    linewidth=0.5,
    # alpha=0.7,
    orient='h'        # horizontal orientation
)

# -----------------------------------------------------------
# 3. Compute label positions dynamically.
# -----------------------------------------------------------
# Use max R^2 to position labels on the right.
x_max = df_plot['R2_Score'].max()
text_x_pos = x_max-0.25  # text to the right of the longest bar

# Horizontal divider positions (seaborn uses 0, 1, 2... y indices).
sep1 = len(synergy_group) - 0.5
sep2 = len(synergy_group) + len(micro_group) - 0.5

# Draw horizontal dividers.
plt.axhline(y=sep1, color='gray', linestyle=':', linewidth=1.5, alpha=0.8)
plt.axhline(y=sep2, color='gray', linestyle=':', linewidth=1.5, alpha=0.8)

# Group-name labels on the right.
# Group 1: Synergistic
plt.text(x=text_x_pos, y=(len(synergy_group)-1)/2, 
         s='Synergistic gains\n(Fusion > Single)', 
         ha='left', va='center', fontsize=14, fontweight='bold', color='#333')

# Group 2: Micro
plt.text(x=text_x_pos, y=sep1 + len(micro_group)/2, 
         s='Micro-scale dominant\n(Fusion ≈ SV)', 
         ha='left', va='center', fontsize=14, fontweight='bold', color='#555')

# Group 3: Macro
plt.text(x=text_x_pos, y=sep2 + len(macro_group)/2, 
         s='Macro-scale dominant\n(Fusion ≈ Satellite)', 
         ha='left', va='center', fontsize=14, fontweight='bold', color='#555')

# -----------------------------------------------------------
# 5. Layout polish.
# -----------------------------------------------------------
plt.xlabel('$R^2$', fontsize=14, fontweight='bold')
plt.ylabel('Sustainable Development Goals (SDG)', fontsize=14, fontweight='bold')
ax.tick_params(axis='both', which='major', labelsize=12)

# Widen x-range to make room for the right-side labels.
plt.xlim(0, x_max) 

# Place legend at lower-right or wherever fits.
plt.legend(title=None, fontsize=12, loc='upper right', frameon=False)

# Hide top and right spines.
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

plt.tight_layout()
# plt.savefig('../../data/figure_assets/fig2_modal_compare.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.lines import Line2D # for custom legend handles

In [ ]:
# === 1. Color palette ===
custom_palette = {
    'Fusion (Ours)':    '#8E8BFE', # primary purple
    'Street view only': '#5FB7B9', # teal
    'Satellite only':   '#F9C74F'  # yellow
}

# Coerce to str so int/str labels match consistently.
df_plot['SDG'] = df_plot['SDG'].astype(str)
plot_order = [str(x) for x in plot_order]

# === 3. Set up canvas ===
fig, ax = plt.subplots(figsize=(5, 10))
fixed_err = 0.02 # placeholder errorbar if std data is missing

# === 4. Plot loop ===
for i, sdg in enumerate(plot_order):
    # Data-filter logic unchanged.
    current_data = df_plot[df_plot['SDG'] == sdg]
    if len(current_data) == 0: continue

    # Extractor helpers unchanged.
    def get_r2(metric_name):
        row = current_data[current_data['Metric'] == metric_name]
        return row['R2_Score'].mean(), row['R2_Score'].std() if not row.empty else None

    val_fusion, std_fusion = get_r2('Fusion (Ours)')
    val_sv, std_sv = get_r2('Street view only')
    val_rs, std_rs = get_r2('Satellite only')

    # 1. Min/max of the current row (for connector line).
    valid_vals = [v for v in [val_fusion, val_sv, val_rs] if v is not None]
    
    if valid_vals:
        min_val = min(valid_vals)
        max_val = max(valid_vals)
        
        # Connector line from leftmost to rightmost point.
        # color='#E0E0E0' (light gray), zorder=0 (bottom layer)
        ax.hlines(y=i, xmin=min_val, xmax=max_val, color='#BDBDBD', linewidth=1.5, zorder=0)

    # =======================================================
    # Point-plotting logic below is unchanged.
    # =======================================================
    if val_rs is not None:
        ax.errorbar(x=val_rs, y=i, xerr=std_rs, fmt='o', 
                    color=custom_palette['Satellite only'], ecolor=custom_palette['Satellite only'],
                    elinewidth=1.5, capsize=3, markersize=12, markeredgecolor='white', markeredgewidth=1, zorder=1)
        
    if val_sv is not None:
        ax.errorbar(x=val_sv, y=i, xerr=std_sv, fmt='o', 
                    color=custom_palette['Street view only'], ecolor=custom_palette['Street view only'],
                    elinewidth=1.5, capsize=3, markersize=12, markeredgecolor='white', markeredgewidth=1, zorder=2)
        
    if val_fusion is not None:
        ax.errorbar(x=val_fusion, y=i, xerr=std_fusion, fmt='*', 
                    color=custom_palette['Fusion (Ours)'], ecolor=custom_palette['Fusion (Ours)'],
                    elinewidth=1.5, capsize=3, markersize=20, markeredgecolor='white', markeredgewidth=0.8, zorder=3)
        
# === 5. Annotation + polish ===
# Divider positions.
sep1 = len(synergy_group) - 0.5
sep2 = len(synergy_group) + len(micro_group) - 0.5
ax.axhline(y=sep1, color='gray', linestyle=':', linewidth=1, alpha=0.6)
ax.axhline(y=sep2, color='gray', linestyle=':', linewidth=1, alpha=0.6)

# Position text dynamically (right of the largest point).
x_max_val = df_plot['R2_Score'].max()
text_x_pos = x_max_val - 0.25

# Add explanatory text.
ax.text(x=text_x_pos, y=(len(synergy_group)-1)/2, s='Synergistic gains\n(Fusion > Single)', 
        ha='left', va='center', fontsize=12, fontweight='bold', color=custom_palette['Fusion (Ours)'])
ax.text(x=text_x_pos, y=sep1 + len(micro_group)/2, s='Micro-scale dominant\n(Fusion ≈ SV)', 
        ha='left', va='center', fontsize=12, color='#555')
ax.text(x=text_x_pos, y=sep2 + len(macro_group)/2, s='Macro-scale dominant\n(Fusion ≈ Satellite)', 
        ha='left', va='center', fontsize=12, color='#555')

# Axis setup.
ax.set_yticks(range(len(plot_order)))
ax.set_yticklabels(plot_order, fontsize=14, fontfamily='Arial')
ax.set_xlabel('$R^2$', fontsize=16, fontweight='bold', fontfamily='Arial')
ax.set_xlim(0, 1) # leave space on the right

# Hide outer frame.
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_linewidth(1)

# Legend.
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=custom_palette['Satellite only'], markersize=10, label='Satellite only'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=custom_palette['Street view only'], markersize=10, label='Street view only'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor=custom_palette['Fusion (Ours)'], markersize=15, label='Fusion (Ours)'),
]
ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=3, frameon=False, fontsize=12)

plt.tight_layout()
plt.savefig('../../data/figure_assets/fig2_modal_compare.svg')
plt.show()